# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohamedahmed02/Flyrank-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

*One row represents one content item for one client on one report date. For this lane, the decision window is March 2026, with the outcome measured in April 2026.*


In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# from google.colab import userdata
# from huggingface_hub import HfApi

# HF_TOKEN = userdata.get("HF_TOKEN")

# api = HfApi(token=HF_TOKEN)

# print("HF token loaded successfully")
# print("Connected to Hugging Face as:", api.whoami()["name"])

One row = one content item for one client on one report date. The working window is March 2026 (2026-03-01 to 2026-03-31).


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

*Features: gsc_clicks, gsc_impressions, gsc_avg_position, ga4_sessions, scroll_events — all measured during March 2026.

Label: label, where 1 means April 2026 organic clicks were higher than March 2026 organic clicks, and 0 otherwise.

Context: client_hash_id, content_hash_id, report_date, month, and data-availability flags.

Excluded: April performance fields and any label-derived fields are excluded from features because they are only known after the decision moment and would cause target leakage.*




In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

grain_check = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS distinct_grain_rows,
        COUNT(*) - COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS duplicate_rows
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
""").df()

grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,distinct_grain_rows,duplicate_rows
0,9841378,9841378,0


In [14]:
window_check = con.execute("""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS min_report_date,
        MAX(report_date) AS max_report_date
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
""").df()

window_check

,row_count,min_report_date,max_report_date
0,9841378,2026-03-01,2026-03-31


In [15]:
availability_check = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
""").df()

availability_check

,total_rows,gsc_available_rows,ga4_available_rows
0,9841378,3611061,413966


In [19]:
feature_frame = con.execute("""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_clicks) AS gsc_clicks,
        SUM(gsc_impressions) AS gsc_impressions,
        AVG(gsc_avg_position) AS gsc_avg_position,
        SUM(ga4_sessions) AS ga4_sessions,
        SUM(scroll_events) AS scroll_events

    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
    GROUP BY
        client_hash_id,
        content_hash_id
),

april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS april_gsc_clicks

    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet'
    )
    WHERE gsc_data_available IS TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    march.client_hash_id,
    march.content_hash_id,
    march.gsc_clicks,
    march.gsc_impressions,
    march.gsc_avg_position,
    march.ga4_sessions,
    march.scroll_events,

    CASE
        WHEN april.april_gsc_clicks > march.gsc_clicks
        THEN 1
        ELSE 0
    END AS label

FROM march
INNER JOIN april
    ON march.client_hash_id = april.client_hash_id
   AND march.content_hash_id = april.content_hash_id
""").df()

feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,gsc_clicks,gsc_impressions,gsc_avg_position,ga4_sessions,scroll_events,label
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,0.0,77.0,4.074107,NaN,NaN,0
1,client_62f4a7e64f5e0096,content_cec711b02f3bbde6,4.0,602.0,4.428747,NaN,NaN,0
2,client_62f4a7e64f5e0096,content_275b6f7f733016d4,1.0,810.0,4.866123,NaN,NaN,0
3,client_62f4a7e64f5e0096,content_ceaec531566ffcfc,0.0,82.0,8.978086,NaN,NaN,0
4,client_62f4a7e64f5e0096,content_755d951187fcd70a,6.0,1858.0,1.854929,NaN,NaN,0


Feature availability:

- gsc_clicks — available at the decision moment because it is measured from GSC data during March 2026.
- gsc_impressions — available at the decision moment because it is measured from GSC data during March 2026.
- gsc_avg_position — available at the decision moment when GSC position data is present for the content.
- ga4_sessions — available at the decision moment when GA4 data is available for the content.
- scroll_events — available at the decision moment when site engagement data is available for the content.

Missing values are retained rather than filled with future information, because availability differs across clients and content.

In [20]:
feature_frame[
    [
        "gsc_clicks",
        "gsc_impressions",
        "gsc_avg_position",
        "ga4_sessions",
        "scroll_events",
        "label"
    ]
].isna().sum()

,0
gsc_clicks,0
gsc_impressions,0
gsc_avg_position,17892
ga4_sessions,48440
scroll_events,48440
label,0


In [21]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

features = [
    "gsc_clicks",
    "gsc_impressions",
    "gsc_avg_position",
    "ga4_sessions",
    "scroll_events"
]

X = feature_frame[features].fillna(0)
y = feature_frame["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

honest_score = roc_auc_score(
    y_test,
    model.predict_proba(X_test)[:, 1]
)

print(f"Honest ROC-AUC: {honest_score:.3f}")

Honest ROC-AUC: 0.668


In [22]:
leaky_features = [
    "gsc_clicks",
    "gsc_impressions",
    "gsc_avg_position",
    "ga4_sessions",
    "scroll_events",
    "label"   # DELIBERATE LEAKAGE
]

X_leaky = feature_frame[leaky_features].fillna(0)

X_train, X_test, y_train, y_test = train_test_split(
    X_leaky,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

leaky_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

leaky_model.fit(X_train, y_train)

leaky_score = roc_auc_score(
    y_test,
    leaky_model.predict_proba(X_test)[:, 1]
)

print(f"Leaky ROC-AUC: {leaky_score:.3f}")

Leaky ROC-AUC: 1.000


In [23]:
# Remove the leaked label and keep only information
# that is available at the decision moment.

honest_features = [
    "gsc_clicks",
    "gsc_impressions",
    "gsc_avg_position",
    "ga4_sessions",
    "scroll_events"
]

print("Honest features:")
print(honest_features)

print(f"\nHonest ROC-AUC: {honest_score:.3f}")
print(f"Leaky ROC-AUC: {leaky_score:.3f}")

Honest features:
['gsc_clicks', 'gsc_impressions', 'gsc_avg_position', 'ga4_sessions', 'scroll_events']

Honest ROC-AUC: 0.668
Leaky ROC-AUC: 1.000


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

One limitation is incomplete data coverage: GA4 and scroll-event features are missing for some content, and GSC average position is also missing for some rows. This means the feature set is not equally available for every content item.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [2]:
!pip -q install duckdb

In [3]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    "CREATE SECRET (TYPE huggingface, TOKEN ?)",
    [HF_TOKEN]
)

print("DuckDB connected to Hugging Face")

DuckDB connected to Hugging Face


In [6]:
rel = "hf://datasets/FlyRank/internship-warehouse"

df = con.sql(f"""
    SELECT *
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    LIMIT 5
""").df()

df

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,2025-01
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,2025-01
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,2025-01
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,2025-01
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,2025-01


In [7]:
schema = con.execute("""
    DESCRIBE
    SELECT *
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
""").df()

schema[["column_name", "column_type"]]

,column_name,column_type
0,report_date,DATE
1,client_hash_id,VARCHAR
2,content_hash_id,VARCHAR
3,client_has_gsc,BOOLEAN
4,client_has_ga4,BOOLEAN
5,gsc_data_available,BOOLEAN
6,ga4_data_available,BOOLEAN
7,gsc_impressions,BIGINT
8,gsc_clicks,BIGINT
9,gsc_sum_position,BIGINT


In [8]:
grain_check = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS distinct_grain_rows,
        COUNT(*) - COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS duplicate_rows
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
""").df()

grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,distinct_grain_rows,duplicate_rows
0,9841378,9841378,0


In [9]:
window_check = con.execute("""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS min_report_date,
        MAX(report_date) AS max_report_date
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
""").df()

window_check

,row_count,min_report_date,max_report_date
0,9841378,2026-03-01,2026-03-31


In [10]:
availability_check = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
""").df()

availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows,ga4_available_rows
0,9841378,3611061,413966


In [11]:
months = con.execute("""
    SELECT
        month,
        COUNT(*) AS rows
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    GROUP BY month
    ORDER BY month
""").df()

months

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,month,rows
0,2025-01,1297
1,2025-02,75985
2,2025-03,167859
3,2025-04,285114
4,2025-05,349923
5,2025-06,329201
6,2025-07,469794
7,2025-08,704962
8,2025-09,845813
9,2025-10,2165471


In [16]:
label_availability = con.execute("""
    WITH march AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_clicks) AS march_clicks
        FROM read_parquet(
            'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
        )
        WHERE gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    ),
    april AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_clicks) AS april_clicks
        FROM read_parquet(
            'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet'
        )
        WHERE gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT
        COUNT(*) AS march_items,
        COUNT(april.content_hash_id) AS items_with_april,
        COUNT(*) - COUNT(april.content_hash_id) AS items_without_april
    FROM march
    LEFT JOIN april
      ON march.client_hash_id = april.client_hash_id
     AND march.content_hash_id = april.content_hash_id
""").df()

label_availability

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,march_items,items_with_april,items_without_april
0,176738,158549,18189


In [17]:
label_check = con.execute("""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS march_clicks
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
),
april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS april_clicks
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet'
    )
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    COUNT(*) AS labeled_rows,
    SUM(CASE WHEN april_clicks > march_clicks THEN 1 ELSE 0 END) AS positive_label,
    SUM(CASE WHEN april_clicks <= march_clicks THEN 1 ELSE 0 END) AS negative_label
FROM march
INNER JOIN april
    ON march.client_hash_id = april.client_hash_id
   AND march.content_hash_id = april.content_hash_id
""").df()

label_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,labeled_rows,positive_label,negative_label
0,158549,27941.0,130608.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.